This notebook is designed to analyze the characteristics of a single sample across a range of temperatures.

The aim of this notebook is to produce a scatter plot of critical fields at certain temperatures

In [46]:
# Import necessary librarys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
from scipy.signal import medfilt
from scipy.ndimage import gaussian_filter1d

In [71]:
# Import mutliple .txt files

# Combine all files for a given sample

# Available samples: (PB2, SN, IN, SNPB, PTFE, IN2BI, IN4BI, CU, PB0, PB1)
# Replace blank in "sc_data/__/*.txt" with sample name

files = glob.glob("meissner/SNPB/*.txt")


# Create dictionary of superconductor files
sc_data = {}

# read files into dictionary
for file in files:
    sc_data[file] = pd.read_csv(file, sep=r"\s+")

Derivative data is unsmoothed

In [ ]:


# Constants
tau = 0.3
cutoff = 5 * tau

for file_path, df_unclean in sc_data.items():
    
    # 1. Separate and Clean Data
    mask = df_unclean["Time(s)"] > cutoff
    df = df_unclean[mask].copy()
    
    # 2. Find split point (Peak of Channel 1)
    max_field_idx = df["Channel_1(V)"].abs().idxmax()
    
    # 3. Split data into segments
    up_sweep = df.loc[:max_field_idx].copy()
    down_sweep = df.loc[max_field_idx:].copy()
    
    # 4. Process UP Sweep
    t_up = up_sweep["Time(s)"]
    kGauss_up = up_sweep["Channel_1(V)"].abs()
    voltage_up = medfilt(up_sweep["Channel_2(V)"].abs(), kernel_size=13)
    
    # 5. Process DOWN Sweep
    t_down = down_sweep["Time(s)"]
    kGauss_down = down_sweep["Channel_1(V)"].abs()
    voltage_down = medfilt(down_sweep["Channel_2(V)"].abs(), kernel_size=13)

    # 6. Calculate dv/dt using numpy gradient
    # This calculates the derivative of voltage with respect to time

    d1v_up = np.gradient(voltage_up, t_up)
    d1v_down = np.gradient(voltage_down, t_down)

    # --- Plotting Section ---

    # Plot 1: Induced Voltage vs kGauss
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1) # Left side plot
    plt.plot(kGauss_up, voltage_up, label='Increasing Field', color='blue')
    plt.plot(kGauss_down, voltage_down, label='Decreasing Field', color='red')
    plt.xlabel("DC Field Strength (kG)")
    plt.ylabel("Induced Voltage (\u03bcV)")
    plt.title(f"Voltage vs Field\n{file_path.split('/')[-1]}")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot 2: dv/dt vs kGauss
    plt.subplot(1, 2, 2) # Right side plot
    plt.plot(kGauss_up, d1v_up, label='Increasing Field', color='blue')
    plt.plot(kGauss_down, d1v_down, label='Decreasing Field', color='red')
    plt.xlabel("DC Field Strength (kG)")
    plt.ylabel("dv/dt (\u03bcV/s)")
    plt.title("dv/dt vs DC Field Strength")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout() # Prevents labels from overlapping
    plt.show()



Derivative data is smoothed

In [ ]:


# Constants
tau = 0.3
cutoff = 5 * tau

for file_path, df_unclean in sc_data.items():
    
    # 1. Separate and Clean Data
    mask = df_unclean["Time(s)"] > cutoff
    df = df_unclean[mask].copy()
    
    # 2. Find split point (Peak of Channel 1)
    max_field_idx = df["Channel_1(V)"].abs().idxmax()
    
    # 3. Split data into segments
    up_sweep = df.loc[:max_field_idx].copy()
    down_sweep = df.loc[max_field_idx:].copy()
    
    # 4. Process UP Sweep
    t_up = up_sweep["Time(s)"]
    kGauss_up = up_sweep["Channel_1(V)"].abs()
    voltage_up = medfilt(up_sweep["Channel_2(V)"].abs(), kernel_size=13)
    
    # 5. Process DOWN Sweep
    t_down = down_sweep["Time(s)"]
    kGauss_down = down_sweep["Channel_1(V)"].abs()
    voltage_down = medfilt(down_sweep["Channel_2(V)"].abs(), kernel_size=13)

    # 6. Calculate dv/dt using numpy gradient
    # This calculates the derivative of voltage with respect to time

    v_gauss_up = gaussian_filter1d(voltage_up, sigma=10)
    v_gauss_down = gaussian_filter1d(voltage_down, sigma=10)

    d1v_up = np.gradient(v_gauss_up, t_up)
    d1v_down = np.gradient(v_gauss_down, t_down)

    # --- Plotting Section ---

    # Plot 1: Induced Voltage vs kGauss
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1) # Left side plot
    plt.plot(kGauss_up, voltage_up, label='Increasing Field', color='blue')
    plt.plot(kGauss_down, voltage_down, label='Decreasing Field', color='red')
    plt.xlabel("DC Field Strength (kG)")
    plt.ylabel("Induced Voltage (\u03bcV)")
    plt.title(f"Voltage vs Field\n{file_path.split('/')[-1]}")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot 2: dv/dt vs kGauss
    plt.subplot(1, 2, 2) # Right side plot
    plt.plot(kGauss_up, d1v_up, label='Increasing Field', color='blue')
    plt.plot(kGauss_down, d1v_down, label='Decreasing Field', color='red')
    plt.xlabel("DC Field Strength (kG)")
    plt.ylabel("dv/dt (\u03bcV/s)")
    plt.title("dv/dt vs DC Field Strength")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout() # Prevents labels from overlapping
    plt.show()
